## project:-Customer Intelligence using Machine Learning Churn Prediction(telco)

### Problem:- the customer are leaving the buisiness or services from company.and we have to stop this and making the model to predict customer churn and also 

#### Data loading

In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv('telecom customers.csv')

In [ ]:
df.head()

#### Data cleaning

In [ ]:
df.isnull().sum()

In [ ]:
 # no null value in data

In [ ]:
df.duplicated().sum()

In [ ]:
# no duplicates in data

In [ ]:
df.info()

#### EDA

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# churn by tenure
sns.boxplot(x=df['Churn'],y=df['tenure'],)

In [ ]:
# from above we can say that the customers is churn that having low tenure means less experience with company 

#### Feature engineering

In [ ]:
# we have multiple categorical columns so we do encoding
# Label encoding for ordinal columns
df2=df.copy()
from sklearn.preprocessing  import LabelEncoder,OneHotEncoder
cat_col=['gender','Partner','Dependents','PhoneService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','PaperlessBilling','Churn']
lbl=LabelEncoder()
for i in cat_col:
    df2[i]=lbl.fit_transform(df2[i])


In [ ]:
# object to int because of some value like ' '
df2['TotalCharges'] = df2['TotalCharges'].astype(str).str.strip()
df2['TotalCharges'] = pd.to_numeric(df2['TotalCharges'], errors='coerce')
# print(df2['TotalCharges'].isnull().sum())
df2['TotalCharges']=df2['TotalCharges'].fillna('0')
df2['TotalCharges']=pd.to_numeric(df2['TotalCharges'])

In [ ]:
# One hot encoding
ohe=OneHotEncoder(drop='first')
cat_col2=df2.select_dtypes('object')
cat_col2=cat_col2.drop('customerID',axis=1)
for j in cat_col2:
    encoded=ohe.fit_transform(df2[[j]])
    data=pd.DataFrame(encoded.toarray(),columns=ohe.get_feature_names_out(),index=df2.index)
    df2=pd.concat([df2,data],axis=1)



In [ ]:
# now we remove the useless columns (feature selection)
# customerID is not affect on churn because it just id and also remove the columns on which we one hot encoded
data_process=df2.drop(columns=['customerID','MultipleLines','InternetService','Contract','PaymentMethod'])
plt.figure(figsize=(30,20))
sns.heatmap(data_process.corr(),annot=True)

In [ ]:
# now we seen all feature visualization which is best feature that affect churn and all thing like this 

### DATA VISUALIZATION AND INSIGHTS

In [ ]:
# tenure and churn
sns.stripplot(x=data_process['Churn'],y=data_process['tenure'])

In [ ]:
# from above we can say that the customer will leave who have less tenure means less experience with company
# strtegy to avoid this:
# we can offer so something for joining and who completed these tenure he got this type of offer to engage the customer

In [ ]:
# totalcharges and churn
sns.boxplot(x=data_process['Churn'],y=data_process['TotalCharges'])

In [ ]:
# from above
# we can say that the customer who churn the company paid less total charges as compare to customers 
# who not leave the company service and here we also say that because of customer who recently join so 
# they paid less charges so we can also say that the tenure and total charges less means there is 
# possibility of customer churn

In [ ]:
# SeniorCitizen vs churn
sns.barplot(x=data_process['Churn'],y=data_process['SeniorCitizen'])

In [ ]:
# from above we can say that the senior citizen having highest churn rate than non senior citizen 
# to avoid these:
# we can give some discount to senior citizen for paying 

In [ ]:
# we can also seen that the tenure distibution is there skweed or not in tenure
sns.kdeplot(data_process['tenure'])

In [ ]:
# from above we say that there is to group because tere is two peak one having high tenure and another having low
# means the churn customer is low as compare to non churn

In [ ]:
data_process['Churn'].value_counts()

In [ ]:
# now we got an best insights that the churn rate is high and we immediately performing some strategy for decrease the churn rate 
# so we need to make a best to best strategy like offers,we can give some day free  StreamingTV to engage the customers


In [ ]:
# so this is buiseness insight and insight now to we build the model that predict the customer churn well 
# we give the info of customer to model and model gives prediction if they say that the customer is churn so 
# we have to apply some strategy on that customer for not churning like some offers

## Model part

In [ ]:
# from above insight and problem we realize that the customer churn is problem so to overcome this 
# we build the model
# we have to correct predict the customer who is churn means true value so the recall is best for model
# evaluation and our data

In [ ]:
x=data_process.drop('Churn',axis=1)
y=data_process['Churn']

In [ ]:
# now train test split
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
# now scaling for better computution ,better performance and same impact
from sklearn.preprocessing import StandardScaler
scl=StandardScaler()
x_train_scaled=scl.fit_transform(x_train)
x_test_scaled=scl.transform(x_test)

### Best Model Selection -recall (because customer churn)

In [ ]:
# logistic regression
from sklearn.linear_model import LogisticRegression
log=LogisticRegression()
log.fit(x_train_scaled,y_train)
log_pred=log.predict(x_test_scaled)
from sklearn.metrics import recall_score,accuracy_score
print('log_recall_score',recall_score(y_test,log_pred))

In [ ]:
# randomforest
from sklearn.ensemble import RandomForestClassifier
rnd=RandomForestClassifier(n_estimators=100)
rnd.fit(x_train_scaled,y_train)
rnd_pred=rnd.predict(x_test_scaled)
print("rnd_recall",recall_score(y_test,rnd_pred))


In [ ]:
# naive bayes
from sklearn.naive_bayes import GaussianNB
gnb=GaussianNB()
gnb.fit(x_train_scaled,y_train)
gnb_pred=gnb.predict(x_test_scaled)
print('gnb_pred',recall_score(y_test,gnb_pred))

In [ ]:
from sklearn.svm import SVC
svc=SVC()
svc.fit(x_train_scaled,y_train)
svc_pred=svc.predict(x_test_scaled)
print('svc_pred',recall_score(y_test,svc_pred))

In [ ]:
from xgboost import XGBClassifier
xgb=XGBClassifier()
xgb.fit(x_train_scaled,y_train)
xgb_pred=xgb.predict(x_test_scaled)
print('xgb_pred',recall_score(y_test,xgb_pred))

In [ ]:
# from all above model we can say that our naive bayes is best for recall
# so we select that

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score,classification_report,confusion_matrix
gnb=GaussianNB()
gnb.fit(x_train_scaled,y_train)
gnb_pred=gnb.predict(x_test_scaled)
print('accuracy_score',accuracy_score(y_test,gnb_pred))
print('recall_score',recall_score(y_test,gnb_pred))
print('precision_score',precision_score(y_test,gnb_pred))
print('f1_score',f1_score(y_test,gnb_pred))
print('confusion_matrix\n',confusion_matrix(y_test,gnb_pred))
print('classification_report\n',classification_report(y_test,gnb_pred))

In [ ]:
# we require only best recall score because its must for customer churn

## Business Recommendations

1. Offer discounts or loyalty benefits to customers with month-to-month contracts, as they have a higher churn rate.

2. Create special retention plans for senior citizens if they are more likely to churn according to the analysis.

3. Provide personalized retention offers (free trial, one month free service, or bill discounts) to customers identified as high-risk by the model.

4. Offer bundled entertainment or streaming services to customer segments where the analysis suggests these services improve retention.

5. Use the churn prediction model to proactively identify high-risk customers and contact them before they leave.